In [51]:
import pandas as pd
import numpy as np

In [52]:
debt_df = pd.read_csv('Data/african_debt_data.csv')

print(debt_df.head())
print(debt_df.columns)

  ISO3  Country BorrowerType         BorrowerAgency              CreditorName  \
0  DZA  Algeria          NaN  Government of Algeria         World Bank (IBRD)   
1  DZA  Algeria          NaN                    NaN  African Development Bank   
2  DZA  Algeria          NaN                    NaN  Islamic Development Bank   
3  DZA  Algeria          NaN  Government of Algeria         World Bank (IBRD)   
4  DZA  Algeria          NaN  Government of Algeria         World Bank (IBRD)   

  CreditorName_short CreditorGroup             CreditorAgency  \
0            WB-IBRD  Multilateral                       IBRD   
1               AfDB  Multilateral      AfDB Ordinary Capital   
2               IsDB  Multilateral  Islamic Development Bank    
3            WB-IBRD  Multilateral                       IBRD   
4            WB-IBRD  Multilateral                       IBRD   

              CreditorAgencyType    year  ... interest  real_interest  price  \
0  Multilateral development bank  2000.0  

In [53]:
# Filter to rows after 2014 (year > 2014) fro thr debt year
df_debt_post2014 = debt_df[debt_df['year'] < 2015].copy()


print(df_debt_post2014.tail())

      ISO3   Country                   BorrowerType  \
56957  ZWE  Zimbabwe             Central government   
56958  ZWE  Zimbabwe             Central government   
56959  ZWE  Zimbabwe  Recipient State-owned Company   
56960  ZWE  Zimbabwe                            NaN   
56961  ZWE  Zimbabwe             Central government   

                                          BorrowerAgency  \
56957                             Government of Zimbabwe   
56958                             Government of Zimbabwe   
56959  Xuzhou Construction Machinery Group Co., Ltd. ...   
56960                             Government of Zimbabwe   
56961                             Government of Zimbabwe   

                                       CreditorName CreditorName_short  \
56957                                         China              China   
56958                                         China              China   
56959                                         China              China   
56960  Arab 

In [54]:
# Cleaning the debt data
debt_clean = debt_df[[
    'Country', 
    'year', 
    'quarter', 
    'Amount_musd', 
    'CreditorGroup', 
    'instrument_type', 
    'interest', 
    'maturity'
]].copy()

print("\n" + "="*60)
print("Debt Dataset - After Cleaning:")
print("Data Shape: {debt_clean.shape}")

print(debt_clean.head())


Debt Dataset - After Cleaning:
Data Shape: {debt_clean.shape}
   Country    year quarter  Amount_musd CreditorGroup    instrument_type  \
0  Algeria  2000.0      Q3         83.0  Multilateral  Multilateral Loan   
1  Algeria  2000.0      Q4        117.0  Multilateral  Multilateral Loan   
2  Algeria  2000.0      Q4         87.0  Multilateral  Multilateral Loan   
3  Algeria  2000.0      Q4          9.0  Multilateral  Multilateral Loan   
4  Algeria  2000.0      Q4          5.0  Multilateral  Multilateral Loan   

   interest   maturity  
0       NaN  16.000000  
1       NaN  20.000000  
2       NaN        NaN  
3       NaN  14.000000  
4       NaN  13.480769  


In [55]:
# Check for Missing Values
print("MISSING VALUES ANALYSIS")

print("" + "-"*60)
print("Debt Dataset:")
print(debt_clean.isnull().sum())

print("Missing percentage: ")
print((debt_clean.isnull().sum() / len(debt_clean) * 100).round(2))


MISSING VALUES ANALYSIS
------------------------------------------------------------
Debt Dataset:
Country               0
year                 14
quarter             210
Amount_musd          14
CreditorGroup         0
instrument_type       0
interest           4232
maturity           6151
dtype: int64
Missing percentage: 
Country             0.00
year                0.02
quarter             0.37
Amount_musd         0.02
CreditorGroup       0.00
instrument_type     0.00
interest            7.43
maturity           10.80
dtype: float64


In [56]:
print("HANDLING MISSING VALUES")
print("" + "-"*60)

# Debt: Drop rows where Amount_musd is missing
debt_clean_original = len(debt_clean)
debt_clean = debt_clean.dropna(subset=['Amount_musd'])
print(f"Debt - rows dropped (missing Amount_musd): {debt_clean_original - len(debt_clean)}")
print(f"Debt - remaining rows: {len(debt_clean)}")

# Check what's left
print("Remaining missing values:")

print(debt_clean.isnull().sum())

HANDLING MISSING VALUES
------------------------------------------------------------
Debt - rows dropped (missing Amount_musd): 14
Debt - remaining rows: 56965
Remaining missing values:
Country               0
year                  0
quarter             210
Amount_musd           0
CreditorGroup         0
instrument_type       0
interest           4232
maturity           6137
dtype: int64


In [57]:
# Aggregating debt data from from months to years
print("AGGREGATING DEBT DATA (QUARTERLY → ANNUAL)")
print("="*60)

# Aggregate by Country and Year
debt_annual = debt_clean.groupby(['Country', 'year']).agg({
    'Amount_musd': 'sum',           
}).reset_index()

# Rename columns for clarity
debt_annual = debt_annual.rename(columns={
    'Amount_musd': 'total_debt_issued_musd',
    'interest': 'avg_interest_rate',
    'maturity': 'avg_maturity_years'
})

print(f"Debt Annual shape: {debt_annual.shape}")
print(f"\nFirst few rows:")
print(debt_annual.head(10))

AGGREGATING DEBT DATA (QUARTERLY → ANNUAL)


Debt Annual shape: (1208, 3)

First few rows:
   Country    year  total_debt_issued_musd
0  Algeria  2000.0                   301.0
1  Algeria  2001.0                   148.0
2  Algeria  2002.0                   251.0
3  Algeria  2003.0                   319.0
4  Algeria  2004.0                    62.0
5  Algeria  2005.0                   156.0
6  Algeria  2006.0                   136.0
7  Algeria  2007.0                    88.0
8  Algeria  2013.0                   295.0
9  Algeria  2016.0                   981.0


In [58]:
# Reading the African Crises Dataset.
crises_df = pd.read_csv('Data/african_crises.csv')
print(crises_df.head())
print(crises_df.columns)


   case  cc3  country  year  systemic_crisis  exch_usd  \
0     1  DZA  Algeria  1870                1  0.052264   
1     1  DZA  Algeria  1871                0  0.052798   
2     1  DZA  Algeria  1872                0  0.052274   
3     1  DZA  Algeria  1873                0  0.051680   
4     1  DZA  Algeria  1874                0  0.051308   

   domestic_debt_in_default  sovereign_external_debt_default  \
0                         0                                0   
1                         0                                0   
2                         0                                0   
3                         0                                0   
4                         0                                0   

   gdp_weighted_default  inflation_annual_cpi  independence  currency_crises  \
0                   0.0              3.441456             0                0   
1                   0.0             14.149140             0                0   
2                   0.0   

In [59]:
# Lets start of with cleaning the crises data
crises_clean = crises_df[[
    'country', 
    'year', 
    'systemic_crisis', 
    'banking_crisis', 
    'currency_crises', 
    'inflation_crises', 
    'inflation_annual_cpi',
    'sovereign_external_debt_default'
]].copy()

# Filter for 2000-2014 (overlap period with debt data)
crises_clean = crises_clean[(crises_clean['year'] >= 2000) & (crises_clean['year'] <= 2014)]

print("Crises Dataset - After Cleaning:")
print(f"Shape: {crises_clean.shape}")

print(crises_clean.head())


Crises Dataset - After Cleaning:
Shape: (193, 8)
    country  year  systemic_crisis banking_crisis  currency_crises  \
70  Algeria  2000                0      no_crisis                0   
71  Algeria  2001                0      no_crisis                0   
72  Algeria  2002                0      no_crisis                0   
73  Algeria  2003                0      no_crisis                0   
74  Algeria  2004                0      no_crisis                0   

    inflation_crises  inflation_annual_cpi  sovereign_external_debt_default  
70                 0                 0.300                                0  
71                 0                 4.200                                0  
72                 0                 1.430                                0  
73                 0                 4.259                                0  
74                 0                 3.972                                0  


In [60]:
# Check for Missing Values
print("MISSING VALUES ANALYSIS")

print("Crises Dataset:")
print(crises_clean.isnull().sum())

print(f"Missing percentage:")
print((crises_clean.isnull().sum() / len(crises_clean) * 100).round(2))


MISSING VALUES ANALYSIS
Crises Dataset:
country                            0
year                               0
systemic_crisis                    0
banking_crisis                     0
currency_crises                    0
inflation_crises                   0
inflation_annual_cpi               0
sovereign_external_debt_default    0
dtype: int64
Missing percentage:
country                            0.0
year                               0.0
systemic_crisis                    0.0
banking_crisis                     0.0
currency_crises                    0.0
inflation_crises                   0.0
inflation_annual_cpi               0.0
sovereign_external_debt_default    0.0
dtype: float64


In [61]:
print("HANDLING MISSING VALUES")

# Crises: No action needed since there was only one column with missing values which is the inflation column
print(f"Crises rows with missing inflation: {crises_clean['inflation_annual_cpi'].isnull().sum()}")



HANDLING MISSING VALUES
Crises rows with missing inflation: 0


In [62]:
# Check current values
print("="*60)
print("DATA TRANSFORMATIONS")
print("="*60)

print("Banking crisis values before transformation:")
print(crises_clean['banking_crisis'].value_counts())

# Convert to binary (1 = crisis, 0 = no crisis)
crises_clean['banking_crisis_binary'] = (crises_clean['banking_crisis'] == 'crisis').astype(int)

# Drop original text column
crises_clean = crises_clean.drop('banking_crisis', axis=1)

print("Banking crisis after transformation:")
print(crises_clean['banking_crisis_binary'].value_counts())

DATA TRANSFORMATIONS
Banking crisis values before transformation:
banking_crisis
no_crisis    177
crisis        16
Name: count, dtype: int64
Banking crisis after transformation:
banking_crisis_binary
0    177
1     16
Name: count, dtype: int64


In [63]:
# Finding the Common Countries between the two datasets
print("FINDING COMMON COUNTRIES")
print("="*60)

crises_countries = set(crises_clean['country'].unique())
debt_countries = set(debt_annual['Country'].unique())

common_countries = crises_countries & debt_countries

print(f"Countries in Crises dataset: {len(crises_countries)}")
print(sorted(crises_countries))

print(f"Countries in Debt dataset: {len(debt_countries)}")
print(sorted(debt_countries))

print(f"Common countries: {len(common_countries)}")
print(sorted(common_countries))

# Check for naming mismatches
only_crises = crises_countries - debt_countries
only_debt = debt_countries - crises_countries

if only_crises:
    print(f"Only in Crises (potential name mismatches): {sorted(only_crises)}")
if only_debt:
    print(f"Only in Debt: {sorted(only_debt)}")

FINDING COMMON COUNTRIES
Countries in Crises dataset: 13
['Algeria', 'Angola', 'Central African Republic', 'Egypt', 'Ivory Coast', 'Kenya', 'Mauritius', 'Morocco', 'Nigeria', 'South Africa', 'Tunisia', 'Zambia', 'Zimbabwe']
Countries in Debt dataset: 54
['Algeria', 'Angola', 'Benin', 'Botswana', 'Burkina Faso', 'Burundi', 'Cameroon', 'Cape Verde', 'Central African Republic', 'Chad', 'Comoros', 'Congo', "Cote d'Ivoire", 'Democratic Republic of Congo', 'Djibouti', 'Egypt', 'Equatorial Guinea', 'Eritrea', 'Ethiopia', 'Gabon', 'Gambia', 'Ghana', 'Guinea', 'Guinea-Bissau', 'Kenya', 'Lesotho', 'Liberia', 'Libya', 'Madagascar', 'Malawi', 'Mali', 'Mauritania', 'Mauritius', 'Morocco', 'Mozambique', 'Namibia', 'Niger', 'Nigeria', 'Rwanda', 'Sao Tome and Principe', 'Senegal', 'Seychelles', 'Sierra Leone', 'Somalia', 'South Africa', 'South Sudan', 'Sudan', 'Swaziland', 'Tanzania', 'Togo', 'Tunisia', 'Uganda', 'Zambia', 'Zimbabwe']
Common countries: 12
['Algeria', 'Angola', 'Central African Republi

In [64]:
# Save cleaned datasets
crises_clean.to_csv('crises_cleaned.csv', index=False)
debt_annual.to_csv('debt_annual_cleaned.csv', index=False)

print("DATASETS SAVED")
print("="*60)
print("✓ crises_cleaned.csv")
print("✓ debt_annual_cleaned.csv")

DATASETS SAVED
✓ crises_cleaned.csv
✓ debt_annual_cleaned.csv


In [65]:
# Found out that Cote,d,Ivore and Ovory Coast are the same country just different naming conventions

# Replacing the Cote d'Ivoire in the debt data to Ivory Coast
debt_annual['Country'] = debt_annual['Country'].replace({"Cote d'Ivoire": "Ivory Coast"})


crises_countries = set(crises_clean['country'].unique())
debt_countries = set(debt_annual['Country'].unique())
common_countries = crises_countries & debt_countries

print(f"Common countries after fix: {len(common_countries)}")
print(sorted(common_countries))



Common countries after fix: 13
['Algeria', 'Angola', 'Central African Republic', 'Egypt', 'Ivory Coast', 'Kenya', 'Mauritius', 'Morocco', 'Nigeria', 'South Africa', 'Tunisia', 'Zambia', 'Zimbabwe']


In [66]:
print("MERGING DATASETS")
print("="*60)

# Filter for common countries only
crises_filtered = crises_clean[crises_clean['country'].isin(common_countries)].copy()
debt_filtered = debt_annual[debt_annual['Country'].isin(common_countries)].copy()

print(f"Crises dataset (filtered): {crises_filtered.shape}")
print(f"Debt dataset (filtered): {debt_filtered.shape}")

# Merge on country and year
merged_df = pd.merge(
    debt_filtered,
    crises_filtered,
    left_on=['Country', 'year'],
    right_on=['country', 'year'],
    how='inner'  
)

# Drop duplicate country column
merged_df = merged_df.drop('country', axis=1)

print(f"Merged dataset shape: {merged_df.shape}")
print(f"Years covered: {merged_df['year'].min()} to {merged_df['year'].max()}")
print(f"Countries: {merged_df['Country'].nunique()}")

print(merged_df.head(10))

print(f"\nColumns in merged dataset:")
print(merged_df.columns.tolist())

MERGING DATASETS
Crises dataset (filtered): (193, 8)
Debt dataset (filtered): (309, 3)
Merged dataset shape: (182, 9)
Years covered: 2000.0 to 2014.0
Countries: 13
   Country    year  total_debt_issued_musd  systemic_crisis  currency_crises  \
0  Algeria  2000.0                   301.0                0                0   
1  Algeria  2001.0                   148.0                0                0   
2  Algeria  2002.0                   251.0                0                0   
3  Algeria  2003.0                   319.0                0                0   
4  Algeria  2004.0                    62.0                0                0   
5  Algeria  2005.0                   156.0                0                0   
6  Algeria  2006.0                   136.0                0                0   
7  Algeria  2007.0                    88.0                0                0   
8  Algeria  2013.0                   295.0                0                0   
9   Angola  2000.0                  

In [67]:
# Final Check for Missing values
print("MISSING VALUES IN MERGED DATASET")
print("="*60)

print(merged_df.isnull().sum())
print("Missing percentages: ")
print((merged_df.isnull().sum() / len(merged_df) * 100).round(2))



MISSING VALUES IN MERGED DATASET
Country                            0
year                               0
total_debt_issued_musd             0
systemic_crisis                    0
currency_crises                    0
inflation_crises                   0
inflation_annual_cpi               0
sovereign_external_debt_default    0
banking_crisis_binary              0
dtype: int64
Missing percentages: 
Country                            0.0
year                               0.0
total_debt_issued_musd             0.0
systemic_crisis                    0.0
currency_crises                    0.0
inflation_crises                   0.0
inflation_annual_cpi               0.0
sovereign_external_debt_default    0.0
banking_crisis_binary              0.0
dtype: float64


In [68]:
# Summary statistics
print("MERGED DATASET SUMMARY")
print("="*60)
print(merged_df.describe())

MERGED DATASET SUMMARY
              year  total_debt_issued_musd  systemic_crisis  currency_crises  \
count   182.000000              182.000000       182.000000       182.000000   
mean   2006.785714             7533.860440         0.082418         0.137363   
std       4.314080            23480.034179         0.275758         0.345179   
min    2000.000000                2.000000         0.000000         0.000000   
25%    2003.000000              122.250000         0.000000         0.000000   
50%    2007.000000              928.500000         0.000000         0.000000   
75%    2010.750000             2504.350000         0.000000         0.000000   
max    2014.000000           188113.500000         1.000000         1.000000   

       inflation_crises  inflation_annual_cpi  \
count        182.000000          1.820000e+02   
mean           0.104396          1.212131e+05   
std            0.306617          1.629963e+06   
min            0.000000         -2.244000e+00   
25%        

In [69]:
# Save the merged dataset
merged_df.to_csv('debt_crisis_merged_final.csv', index=False)

print("✓ FINAL MERGED DATASET SAVED")
print("="*60)
print("File: debt_crisis_merged_final.csv")
print(f"Shape: {merged_df.shape}")
print(f"Countries: {merged_df['Country'].nunique()}")
print(f"Years: {merged_df['year'].min()} - {merged_df['year'].max()}")
print(f"Total observations: {len(merged_df)}")

✓ FINAL MERGED DATASET SAVED
File: debt_crisis_merged_final.csv
Shape: (182, 9)
Countries: 13
Years: 2000.0 - 2014.0
Total observations: 182
